### Animal Faces Classification Task

In [1]:
# Import necessary libraries
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
# from tensorflow.keras.preprocessing.image import ImageDataGenerator
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
# from tensorflow.keras.regularizers import l1, l2
# from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.callbacks import EarlyStopping
import kagglehub
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
# Set random seed for reproducibility
np.random.seed(42)
# tf.random.set_seed(42)
torch.manual_seed(42)

generator = torch.Generator().manual_seed(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [2]:
# Download the dataset
path = kagglehub.dataset_download("andrewmvd/animal-faces")
print("Path to dataset files:", path)

# Define paths (assuming standard structure: afhq/train and afhq/val with subfolders cat, dog, wild)
data_dir = os.path.join(path, 'afhq')
train_dir = os.path.join(data_dir, 'train')
test_dir = os.path.join(data_dir, 'val')  # This is the test set in this task

num_classes = 3  # cat, dog, wild

batch_size = 64

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])

])

train_dataset = datasets.ImageFolder(train_dir, transform=transform)
test_dataset = datasets.ImageFolder(test_dir, transform=transform)
train_size = len(train_dataset)
indices = np.arange(train_size)
np.random.shuffle(indices)
val_size = int(0.05 * train_size)
val_set = torch.utils.data.Subset(train_dataset, list(indices[:val_size]))
train_set = torch.utils.data.Subset(train_dataset, list(indices[val_size:])) 
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
train_generator = iter(train_loader)
test_generator = iter(test_loader)

# Helpful info
print(f"Found {len(train_dataset)} training images in {len(train_dataset.classes)} classes: {train_dataset.classes}")
print(f"Found {len(test_dataset)} test images in {len(test_dataset.classes)} classes: {test_dataset.classes}")

# Note: The 'val' folder is used as the test set here. If you need a validation set, split the data from the train set.

Path to dataset files: C:\Users\arman\.cache\kagglehub\datasets\andrewmvd\animal-faces\versions\1
Found 14630 training images in 3 classes: ['cat', 'dog', 'wild']
Found 1500 test images in 3 classes: ['cat', 'dog', 'wild']


In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 32, 3)
        # self.flatten1 = nn.Flatten()
        self.fc1 = nn.Linear(32 * 14 *14, 7317)
        self.fc2 = nn.Linear(7317, 3)
        # self.fc3 = nn.Linear(1024, 3)
        # self.dropout1 = nn.Dropout(0.2)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        # x = self.fc3(x)
        # x = F.softmax(x,dim=1)
        return x


net = Net()

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(),lr=0.0001)

In [5]:
num_epochs = 5

for epoch in range(num_epochs):
    net.train()
    correct = 0
    total = 0
    train_loss = 0.0
    for images,labels in train_loader:
        outputs = net(images)
        loss = criterion(outputs,labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    net.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = net(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_accuracy = 100 * correct_val / total_val
    print(f'Epoch : {epoch + 1}, loss : {loss.item()}, train acc : {correct / total * 100}, val acc: {val_accuracy}')

Epoch : 1, loss : 0.4072408080101013, train acc : 81.46629253903158, val acc: 89.60328317373461
Epoch : 2, loss : 0.21114061772823334, train acc : 90.79070436722067, val acc: 90.42407660738714
Epoch : 3, loss : 0.08677417039871216, train acc : 93.34484495287431, val acc: 91.92886456908344
Epoch : 4, loss : 0.13505247235298157, train acc : 95.17950931721707, val acc: 90.8344733242134
Epoch : 5, loss : 0.3723744750022888, train acc : 95.90618030074106, val acc: 92.20246238030096


In [6]:
def measure_accuracy(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    print(f'Accuracy: {accuracy:.2f}%')
    return accuracy

In [7]:
measure_accuracy(net,test_loader)

Accuracy: 94.00%


94.0